In [ ]:
#https://scikit-learn.org/stable/modules/feature_selection.html

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
abandono = pd.read_csv('./Abandono/abandono.csv',index_col=0)
abandono

In [ ]:
grades_per_exam = pd.read_csv('./Grades/grades_per_exam.csv',index_col=0)
grades_per_exam

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn import svm

In [ ]:
aucs = {}
w = 7
indicadores =  pd.read_csv('./Modelos/Cummulative/Indicadores/indicadores_week1_{}.csv'.format(w),index_col=0)
theta = pd.read_csv('./Modelos/Cummulative/Configuración al menos un evaluable/theta_one_week{}.csv'.format(w),index_col=0)
w_ind = indicadores.loc[theta.index]
w_ind['theta'] = theta['conf1']
w_ind['passed'] = abandono['curso_superado']
w_ind['dropped_out'] = abandono['abandono']

w_ind['CFA'] = w_ind['CFA'].replace('0 problems attempted',0)
w_ind['avg_grade'] = w_ind['avg_grade'].replace('0 problems attempted',0)
w_ind['avg_attempts'] = w_ind['avg_attempts'].replace('0 problems attempted',0)
w_ind['dropped_out'] = w_ind['dropped_out'].replace('No',0).replace('Sí',1)
w_ind['passed'] = w_ind['passed'].replace('No',0).replace('Sí',1)

X = w_ind.drop(['dropped_out','passed'],axis=1)

In [ ]:
X

**Technique 1: Variance Threshold**

Quitamos variable: per_videos. En abandono, las curvas AUCs aumentan ~ 0.01

https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.VarianceThreshold.html#sklearn.feature_selection.VarianceThreshold

In [ ]:
# VarianceThreshold: Removes features with low variance.
from sklearn.feature_selection import VarianceThreshold
sel = VarianceThreshold(threshold=(.95 * (1 - .95))) # Remove all variables that are zero in more than 80% of the samples
X_var = sel.fit_transform(X)

In [ ]:
X_var.shape

In [ ]:
X.columns

In [ ]:
sel.get_feature_names_out()

In [ ]:
X_new = X[['n_attempted', 'per_attempted', 'CFA', 'avg_grade', 'avg_attempts',
       'n_show', 'n_pauses', 'n_threads', 'n_responses', 'n_comments',
       'n_days', 'n_videos_watched', 'n_videos_opened', 'theta']]
y = w_ind['dropped_out']

X_train, X_test, y_train, y_test = train_test_split(X_new, y, test_size=0.25, random_state=42)

param_range = [1, 2, 3, 5]
param_range_fl = [1.0, 0.5, 0.1]
n_estimators = [50,75,100]
learning_rates = [.1,.2,.3]

# Logistic Regression
pipe_lr = Pipeline([('scl', StandardScaler()),
                  ('LR', LogisticRegression(random_state=42))])
lr_param_grid = [{'LR__penalty': ['l1', 'l2'],
                  'LR__C': param_range_fl,
                  'LR__solver': ['liblinear']}]
lr_grid_search = GridSearchCV(estimator=pipe_lr,
        param_grid=lr_param_grid,
        scoring='roc_auc',
        cv=3)
lr_grid_search.fit(X_train,y_train)
print('Logistic Regression:', lr_grid_search.score(X_test,y_test))

# Random Forest
pipe_svm = Pipeline([('scl', StandardScaler()),
                    ('SVM', svm.SVC(random_state=42))])
svm_param_grid = [{'SVM__kernel': ['linear', 'rbf'],
                      'SVM__C': param_range}]
svm_grid_search = GridSearchCV(estimator=pipe_svm,
          param_grid=svm_param_grid,
          scoring='roc_auc',
          cv=3)
svm_grid_search.fit(X_train,y_train)
print('Support Vector Machine:', svm_grid_search.score(X_test,y_test))

# Decision Trees
pipe_dt = Pipeline([('scl', StandardScaler()),
                    ('DT',DecisionTreeClassifier(random_state=42))])
dt_param_grid = [{'DT__criterion': ['gini', 'entropy'],
                    'DT__min_samples_leaf': param_range,
                    'DT__max_depth': param_range,
                    'DT__min_samples_split': param_range[1:]}]
dt_grid_search = GridSearchCV(estimator=pipe_dt,
param_grid=dt_param_grid,
        scoring='roc_auc',
        cv=3)
dt_grid_search.fit(X_train,y_train)
print('Decision Trees:', dt_grid_search.score(X_test,y_test))

# XGBoost
pipe_xgb = Pipeline([('scl', StandardScaler()),
                    ('XGB', XGBClassifier())])
xgb_param_grid = [{'XGB__learning_rate': learning_rates,
                      'XGB__max_depth': param_range,
                      'XGB__min_child_weight': param_range[:2],
                      'XGB__subsample': param_range_fl,
                      'XGB__n_estimators': n_estimators}]
xgb_grid_search = GridSearchCV(estimator=pipe_xgb,
        param_grid=xgb_param_grid,
        scoring='roc_auc',
        cv=3)

xgb_grid_search.fit(X_train,y_train)
print('XGBoost:', xgb_grid_search.score(X_test,y_test))

In [ ]:
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
sel2 = SelectKBest(f_classif, k=5)
X_new2 = sel2.fit_transform(X, y)
X_new2.shape

In [ ]:
X_new2

In [ ]:
sel2.get_feature_names_out()

In [ ]:
aucs = {}
w = 7
indicadores =  pd.read_csv('./Modelos/Cummulative/Indicadores/indicadores_week1_{}.csv'.format(w),index_col=0)
theta = pd.read_csv('./Modelos/Cummulative/Configuración al menos un evaluable/theta_one_week{}.csv'.format(w),index_col=0)
w_ind = indicadores.loc[theta.index]
w_ind['theta'] = theta['conf1']
w_ind['passed'] = abandono['curso_superado']
w_ind['dropped_out'] = abandono['abandono']

w_ind['CFA'] = w_ind['CFA'].replace('0 problems attempted',0)
w_ind['avg_grade'] = w_ind['avg_grade'].replace('0 problems attempted',0)
w_ind['avg_attempts'] = w_ind['avg_attempts'].replace('0 problems attempted',0)
w_ind['dropped_out'] = w_ind['dropped_out'].replace('No',0).replace('Sí',1)
w_ind['passed'] = w_ind['passed'].replace('No',0).replace('Sí',1)

X = w_ind.drop(['dropped_out','passed'],axis=1)
X_new2 = X[['n_attempted', 'per_attempted', 'n_days', 'n_videos_opened',
       'theta']]
y = w_ind['dropped_out']

X_train, X_test, y_train, y_test = train_test_split(X_new2, y, test_size=0.25, random_state=42)

param_range = [1, 2, 3, 5]
param_range_fl = [1.0, 0.5, 0.1]
n_estimators = [50,75,100]
learning_rates = [.1,.2,.3]

# Logistic Regression
pipe_lr = Pipeline([('scl', StandardScaler()),
                  ('LR', LogisticRegression(random_state=42))])
lr_param_grid = [{'LR__penalty': ['l1', 'l2'],
                  'LR__C': param_range_fl,
                  'LR__solver': ['liblinear']}]
lr_grid_search = GridSearchCV(estimator=pipe_lr,
        param_grid=lr_param_grid,
        scoring='roc_auc',
        cv=3)
lr_grid_search.fit(X_train,y_train)
print('Logistic Regression:', lr_grid_search.score(X_test,y_test))

# Random Forest
pipe_svm = Pipeline([('scl', StandardScaler()),
                    ('SVM', svm.SVC(random_state=42))])
svm_param_grid = [{'SVM__kernel': ['linear', 'rbf'],
                      'SVM__C': param_range}]
svm_grid_search = GridSearchCV(estimator=pipe_svm,
          param_grid=svm_param_grid,
          scoring='roc_auc',
          cv=3)
svm_grid_search.fit(X_train,y_train)
print('Support Vector Machine:', svm_grid_search.score(X_test,y_test))

# Decision Trees
pipe_dt = Pipeline([('scl', StandardScaler()),
                    ('DT',DecisionTreeClassifier(random_state=42))])
dt_param_grid = [{'DT__criterion': ['gini', 'entropy'],
                    'DT__min_samples_leaf': param_range,
                    'DT__max_depth': param_range,
                    'DT__min_samples_split': param_range[1:]}]
dt_grid_search = GridSearchCV(estimator=pipe_dt,
param_grid=dt_param_grid,
        scoring='roc_auc',
        cv=3)
dt_grid_search.fit(X_train,y_train)
print('Decision Trees:', dt_grid_search.score(X_test,y_test))
